# **access storage account**

In [0]:

spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

# **Access data from bronze container**

In [0]:
dbutils.fs.ls(f"abfss://bronze@nyctaxidataforproject.dfs.core.windows.net/nyc_trip_2025")


[FileInfo(path='abfss://bronze@nyctaxidataforproject.dfs.core.windows.net/nyc_trip_2025/trip-data/', name='trip-data/', size=0, modificationTime=1788161414000)]

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_trip_type = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load(f"abfss://bronze@nyctaxidataforproject.dfs.core.windows.net/trip_type/")
df_trip_type.display()


trip_type,description
1,Street-hail
2,Dispatch


In [0]:
df_trip_type.display()


In [0]:
df_trip_zone = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load(f"abfss://bronze@nyctaxidataforproject.dfs.core.windows.net/trip_zone/")

In [0]:
df_trip_zone.display()

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
trip= spark.read.format("parquet").option("inferSchema", "true").option("header", "true").option("recursiveFileLookup", "true").load(f"abfss://bronze@nyctaxidataforproject.dfs.core.windows.net/nyc_trip_2025/")

In [0]:
trip.display()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
2,2025-05-01T00:17:04,2025-05-01T00:56:06,N,1,25,216,1,9.34,44.3,1.0,0.5,0.0,0.0,null,1.0,46.8,1,1,0.0,0.0
2,2025-05-01T00:56:16,2025-05-01T01:10:26,N,1,160,129,1,2.95,16.3,1.0,0.5,0.0,0.0,null,1.0,18.8,2,1,0.0,0.0
1,2025-05-01T00:24:49,2025-05-01T00:42:29,N,1,260,179,1,3.0,18.4,1.0,1.5,0.0,0.0,null,1.0,20.9,2,1,0.0,0.0
2,2025-05-01T00:27:11,2025-05-01T00:33:21,N,1,130,216,1,1.61,9.3,1.0,0.5,0.0,0.0,null,1.0,11.8,2,1,0.0,0.0
2,2025-05-01T00:32:59,2025-05-01T00:41:34,N,1,244,151,2,3.44,15.6,1.0,0.5,4.52,0.0,null,1.0,22.62,1,1,0.0,0.0
2,2025-04-30T23:58:57,2025-05-01T00:02:31,N,1,42,41,1,0.66,6.5,1.0,0.5,2.0,0.0,null,1.0,11.0,1,1,0.0,0.0
2,2025-05-01T00:38:03,2025-05-01T00:43:28,N,1,240,265,1,1.63,9.3,1.0,0.5,0.0,0.0,null,1.0,11.8,1,1,0.0,0.0
2,2025-05-01T00:13:48,2025-05-01T00:26:19,N,1,129,70,1,2.15,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,0.0
2,2025-05-01T00:08:00,2025-05-01T00:22:00,N,1,244,42,1,2.87,15.6,1.0,0.5,0.0,0.0,null,1.0,18.1,2,1,0.0,0.0
2,2025-05-01T00:48:03,2025-05-01T00:57:01,N,1,75,262,1,1.52,10.7,1.0,0.5,2.39,0.0,null,1.0,18.34,1,1,2.75,0.0


# **Data Transformation**

In [0]:
df_trip_type=df_trip_type.withColumnRenamed("description","trip_description")
df_trip_type.display()


trip_type,trip_description
1,Street-hail
2,Dispatch


### save data in silver container

In [0]:
df_trip_type.write.format("parquet").mode('append').option("path","abfss://silver@nyctaxidataforproject.dfs.core.windows.net/trip_type/").save()

In [0]:
df_trip_zone.display()

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:

df_trip_zone=df_trip_zone.withColumn("",get(split(col("Zone"), "/")))



In [0]:
df_trip_zone.display()

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
trip=trip.withColumnRenamed("Date",to_date("lpep_pickup_datetime"))

In [0]:
trip=trip.withColumnRenamed("Year",year("lpep_pickup_datetime"))

In [0]:
trip=trip.withColumnRenamed("Month",month("lpep_pickup_datetime"))

In [0]:
from pyspark.sql.functions import month, to_date, year

trip = (
    spark.read.format("parquet")
    .option("recursiveFileLookup", "true")
    .load("abfss://bronze@nyctaxidataforproject.dfs.core.windows.net/nyc_trip_2025/")
    .withColumn("Date", to_date("lpep_pickup_datetime"))
    .withColumn("Year", year("lpep_pickup_datetime"))
    .withColumn("Month", month("lpep_pickup_datetime"))
)

trip.write.format("parquet").mode("append").save("abfss://silver@nyctaxidataforproject.dfs.core.windows.net/trip_data/")
